<a href="https://colab.research.google.com/github/unamiral/AcousticDETR/blob/main/AcousticDETR_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess, time, os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

res = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True)


In [ ]:
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    SAVE_DIR = '/content/drive/MyDrive/AcousticDETR'
else:
    SAVE_DIR = '/tmp/AcousticDETR'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

In [ ]:
# ── Detector ──────────────────────────────────────────────────────────────────
N_SHEETS    = 5
M_MICS      = 5
SHEET_SIZE  = 0.30       # m
SHEET_GAP   = 0.05       # m
SHEET_THICK = 1e-3       # m

_r = SHEET_SIZE * 0.35
MIC_XY = np.array([[0.,0.],[_r,0.],[-_r,0.],[0.,_r],[0.,-_r]], dtype=np.float32)

# ── Acoustic ──────────────────────────────────────────────────────────────────
C_SOUND_AL  = 6_320.0    # m/s
PULSE_FREQ  = 200_000.0  # Hz
SAMPLE_RATE = 2_000_000  # Hz
WAVEFORM_T  = 256        # samples

# ── Pileup ────────────────────────────────────────────────────────────────────
MAX_TRACKS      = 4
TRACK_PROB      = [0.20, 0.35, 0.30, 0.15]
MAX_TIME_SPREAD = 20e-6  # s

# ── Bethe-Bloch ───────────────────────────────────────────────────────────────
BETHE_K, RHO_AL, Z_AL, A_AL = 0.307, 2.70, 13, 26.98
PROTON_MASS_MEV = 938.272

# ── Noise ─────────────────────────────────────────────────────────────────────
ADC_NOISE_STD   = 0.002
AMP_NOISE_FRAC  = 0.04
TIMING_JITTER_S = 0.3e-6

# ── Model ─────────────────────────────────────────────────────────────────────
D_MODEL            = 128
N_HEADS            = 4
N_SHEET_LAYERS     = 2
N_INTERSHEET_LAYERS = 3
N_DECODER_LAYERS   = 3
CNN_CH             = [32, 64, D_MODEL]
DROPOUT            = 0.1

# ── Training ──────────────────────────────────────────────────────────────────
N_SAMPLES     = 8_000
N_EPOCHS      = 100
BATCH_SIZE    = 64        # fits T4 comfortably
LR            = 5e-4
WARMUP_EPOCHS = 10
TRAIN_SPLIT   = 0.80
SEED          = 42

# ── Loss ──────────────────────────────────────────────────────────────────────
W_POS, W_ANG, W_EXIST = 10.0, 5.0, 2.0
NO_TRACK_COEFF        = 0.1
EXIST_THRESH          = 0.45

# ── Colab optimisation flags ──────────────────────────────────────────────────
USE_AMP          = True   # FP16 mixed precision (~2x speedup on T4)
USE_COMPILE      = True   # torch.compile PyTorch >= 2.0 (~30% speedup)
GRAD_ACCUM_STEPS = 1      # increase to 2 if OOM with larger models
NUM_WORKERS      = 2
PIN_MEMORY       = True

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
T_AXIS = np.arange(WAVEFORM_T) / SAMPLE_RATE   # shared time axis [s]

print(f"Config OK  DEVICE={DEVICE}  AMP={USE_AMP}  COMPILE={USE_COMPILE}")
print(f"Input shape per event: ({N_SHEETS}, {M_MICS}, {WAVEFORM_T})")

In [ ]:
def beta(T_mev):
    g = 1 + T_mev / PROTON_MASS_MEV
    return math.sqrt(max(1 - 1/g**2, 0))

def bethe_bloch_dedx(T_mev):
    b = beta(max(T_mev, 1.0))
    return (BETHE_K * RHO_AL * Z_AL / A_AL) / b**2

def energy_loss(T_mev):
    return min(bethe_bloch_dedx(T_mev) * SHEET_THICK * 100, T_mev)

def hit_xy(x0, y0, theta, phi, k):
    dz = math.cos(theta)
    if abs(dz) < 1e-12: return np.array([x0, y0])
    t  = (k * SHEET_GAP) / dz
    return np.array([x0 + t*math.sin(theta)*math.cos(phi),
                      y0 + t*math.sin(theta)*math.sin(phi)])

def ricker(t_arr, f0=PULSE_FREQ):
    u   = (math.pi * f0 * t_arr) ** 2
    psi = (1 - 2*u) * np.exp(-u)
    return psi / (np.abs(psi).max() + 1e-12)

def mic_waveforms(hit, e_dep, t_arr, rng):
    W = np.zeros((M_MICS, WAVEFORM_T), dtype=np.float32)
    for m in range(M_MICS):
        d   = max(float(np.linalg.norm(hit - MIC_XY[m])), 1e-3)
        amp = max((e_dep / d) * (1 + rng.normal(0, AMP_NOISE_FRAC)), 0)
        toa = t_arr + d / C_SOUND_AL + rng.normal(0, TIMING_JITTER_S)
        W[m] += (amp * ricker(T_AXIS - toa)).astype(np.float32)
    return W

def sheet_waveforms(hits, e_deps, t_arrs, rng):
    W = np.zeros((M_MICS, WAVEFORM_T), dtype=np.float32)
    for h, e, t in zip(hits, e_deps, t_arrs):
        if e > 1e-6:
            W += mic_waveforms(h, e, t, rng)
    W += rng.normal(0, ADC_NOISE_STD, W.shape).astype(np.float32)
    return W

print("Physics engine ✅")
print(f"  dE/dx @ 200 MeV = {bethe_bloch_dedx(200):.3f} MeV/cm")
print(f"  ΔE per sheet    = {energy_loss(200)*1e3:.2f} keV")

In [ ]:
from joblib import Parallel, delayed
from torch.utils.data import Dataset, DataLoader, random_split

def simulate_one(seed_int):
    rng = np.random.default_rng(seed_int)
    n   = int(rng.choice(np.arange(1, MAX_TRACKS+1), p=TRACK_PROB))
    half = SHEET_SIZE / 2 * 0.85

    tracks = [{'x0':   rng.uniform(-half, half),
               'y0':   rng.uniform(-half, half),
               'theta':rng.uniform(0, np.deg2rad(25)),
               'phi':  rng.uniform(0, 2*np.pi),
               'E':    rng.uniform(80, 600),
               't0':   rng.uniform(0, MAX_TIME_SPREAD)} for _ in range(n)]

    wf      = np.zeros((N_SHEETS, M_MICS, WAVEFORM_T), dtype=np.float32)
    energies = [t['E'] for t in tracks]

    for k in range(N_SHEETS):
        hits, edeps, tarrs = [], [], []
        for i, tr in enumerate(tracks):
            hits.append(hit_xy(tr['x0'], tr['y0'], tr['theta'], tr['phi'], k))
            de = energy_loss(energies[i]); edeps.append(de)
            dz_safe = max(abs(math.cos(tr['theta'])), 1e-9)
            tarrs.append(tr['t0'] + k*SHEET_GAP / dz_safe / 3e8)
            energies[i] = max(energies[i] - de, 0)
        wf[k] = sheet_waveforms(hits, edeps, tarrs, rng)

    labels = np.zeros((MAX_TRACKS, 4), dtype=np.float32)
    mask   = np.zeros(MAX_TRACKS, dtype=bool)
    for i, tr in enumerate(tracks):
        labels[i] = [tr['x0'], tr['y0'], tr['theta'], tr['phi']]
        mask[i]   = True

    return wf, labels, mask, n


class PileupDataset(Dataset):
    def __init__(self, X, y, masks, n_trks):
        self.X      = torch.tensor(X,      dtype=torch.float32)
        self.y      = torch.tensor(y,      dtype=torch.float32)
        self.masks  = torch.tensor(masks,  dtype=torch.bool)
        self.n_trks = torch.tensor(n_trks, dtype=torch.int32)
    def __len__(self):        return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i], self.masks[i], self.n_trks[i]


def build_dataset(n=N_SAMPLES, force=False):
    cache = os.path.join(SAVE_DIR, f'dataset_{n}.npz')
    if os.path.exists(cache) and not force:
        print(f"Loading cached dataset …")
        d = np.load(cache)
        return d['X'], d['y'], d['masks'], d['n_trks']

    print(f"Generating {n} events in parallel …")
    t0 = time.time()
    out = Parallel(n_jobs=-1, prefer='threads')(
        delayed(simulate_one)(i) for i in range(n))
    print(f"Done in {time.time()-t0:.1f} s")
    X, y, masks, n_trks = map(np.array, zip(*[(o[0],o[1],o[2],o[3]) for o in out]))
    np.savez_compressed(cache, X=X.astype(np.float32),
                        y=y.astype(np.float32), masks=masks, n_trks=n_trks)
    print(f"Saved → {cache}  X={X.shape}")
    return X, y, masks, n_trks


X_np, y_np, masks_np, ntrk_np = build_dataset(N_SAMPLES)

full_ds = PileupDataset(X_np, y_np, masks_np, ntrk_np)
n_train = int(len(full_ds) * TRAIN_SPLIT)
n_val   = len(full_ds) - n_train
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED))

kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
          pin_memory=PIN_MEMORY and DEVICE.type=='cuda',
          persistent_workers=NUM_WORKERS > 0)
train_dl = DataLoader(train_ds, shuffle=True,  **kw)
val_dl   = DataLoader(val_ds,   shuffle=False, **kw)

from collections import Counter
print(f"Train={n_train}  Val={n_val}")
print(f"Multiplicity dist: {dict(sorted(Counter(ntrk_np.tolist()).items()))}")
print("Dataset ready ✅")

In [ ]:
class WaveformEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        c = CNN_CH
        self.net = nn.Sequential(
            nn.Conv1d(1,c[0],7,padding=3), nn.BatchNorm1d(c[0]), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(c[0],c[1],5,padding=2), nn.BatchNorm1d(c[1]), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(c[1],c[2],3,padding=1), nn.BatchNorm1d(c[2]), nn.GELU(),
            nn.AdaptiveAvgPool1d(1))
    def forward(self, x):          # (..., T) → (..., D)
        s = x.shape[:-1]; T = x.shape[-1]
        return self.net(x.reshape(-1,1,T)).squeeze(-1).reshape(*s, D_MODEL)


class MicPosEnc(nn.Module):
    def __init__(self):
        super().__init__()
        mic = torch.tensor(MIC_XY / (SHEET_SIZE/2), dtype=torch.float32)
        self.register_buffer('mic', mic)
        self.proj = nn.Sequential(
            nn.Linear(2, D_MODEL//2), nn.GELU(), nn.Linear(D_MODEL//2, D_MODEL))
    def forward(self): return self.proj(self.mic)   # (M, D)


class SheetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.mic_pe = MicPosEnc()
        enc = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, D_MODEL*4, DROPOUT,
                                          batch_first=True, activation='gelu', norm_first=True)
        self.tr   = nn.TransformerEncoder(enc, N_SHEET_LAYERS)
        self.norm = nn.LayerNorm(D_MODEL)
    def forward(self, x):   # (B, N, M, D)
        B, N, M, D = x.shape
        x = x + self.mic_pe()
        return self.norm(self.tr(x.view(B*N,M,D)).mean(1)).view(B,N,D)


class InterSheetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        z    = torch.arange(N_SHEETS).float() * SHEET_GAP
        zn   = z / (z.max()+1e-8)
        pe   = torch.zeros(N_SHEETS, D_MODEL)
        div  = torch.exp(torch.arange(0, D_MODEL, 2).float() * (-math.log(10000)/D_MODEL))
        pe[:,0::2] = torch.sin(zn.unsqueeze(1)*div)
        pe[:,1::2] = torch.cos(zn.unsqueeze(1)*div)
        self.register_buffer('z_pe', pe.unsqueeze(0))   # (1, N, D)
        enc = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, D_MODEL*4, DROPOUT,
                                          batch_first=True, activation='gelu', norm_first=True)
        self.tr = nn.TransformerEncoder(enc, N_INTERSHEET_LAYERS)
    def forward(self, x): return self.tr(x + self.z_pe)   # (B, N, D)


class TrackDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.queries = nn.Embedding(MAX_TRACKS, D_MODEL)
        dec = nn.TransformerDecoderLayer(D_MODEL, N_HEADS, D_MODEL*4, DROPOUT,
                                          batch_first=True, activation='gelu', norm_first=True)
        self.tr   = nn.TransformerDecoder(dec, N_DECODER_LAYERS)
        self.norm = nn.LayerNorm(D_MODEL)
    def forward(self, mem):   # (B, N, D) → (B, K, D)
        B = mem.shape[0]
        q = self.queries.weight.unsqueeze(0).expand(B,-1,-1)
        return self.norm(self.tr(q, mem))


class TrajHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp   = nn.Sequential(nn.Linear(D_MODEL,D_MODEL), nn.GELU(),
                                    nn.Linear(D_MODEL,D_MODEL//2), nn.GELU())
        self.pos   = nn.Linear(D_MODEL//2, 2)   # x0, y0
        self.ang   = nn.Linear(D_MODEL//2, 2)   # theta, phi
        self.exist = nn.Linear(D_MODEL//2, 1)
        nn.init.constant_(self.exist.bias, -2.0)   # default: no track
    def forward(self, q):
        h   = self.mlp(q)
        pos = torch.tanh(self.pos(h)) * (SHEET_SIZE/2)
        a   = self.ang(h)
        ang = torch.cat([torch.sigmoid(a[...,0:1])*(math.pi/2), a[...,1:2]], -1)
        return {'pos': pos, 'ang': ang, 'exist': self.exist(h)}


class AcousticDETR(nn.Module):
    def __init__(self):
        super().__init__()
        self.wf_enc    = WaveformEncoder()
        self.sheet_enc = SheetEncoder()
        self.inter_enc = InterSheetEncoder()
        self.decoder   = TrackDecoder()
        self.head      = TrajHead()

    def forward(self, wf):   # (B, N, M, T) → dict
        e   = self.wf_enc(wf)
        s   = self.sheet_enc(e)
        mem = self.inter_enc(s)
        q   = self.decoder(mem)
        out = self.head(q)
        out['prob'] = torch.sigmoid(out['exist'])
        return out

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    @torch.no_grad()
    def predict(self, wf, thresh=EXIST_THRESH):
        self.eval()
        out = self(wf); B = wf.shape[0]
        results = []
        for b in range(B):
            tracks = []
            for k in range(MAX_TRACKS):
                p = out['prob'][b,k,0].item()
                if p > thresh:
                    tracks.append({'x0':  out['pos'][b,k,0].item(),
                                   'y0':  out['pos'][b,k,1].item(),
                                   'theta':out['ang'][b,k,0].item(),
                                   'phi': out['ang'][b,k,1].item(),
                                   'conf': p})
            results.append(tracks)
        return results


model = AcousticDETR().to(DEVICE)
print(f"AcousticDETR ✅  {model.n_params():,} parameters")

# torch.compile — skips gracefully on older PyTorch
if USE_COMPILE and hasattr(torch,'compile') and DEVICE.type=='cuda':
    try:
        model = torch.compile(model, mode='reduce-overhead')
        print("torch.compile")
    except Exception as e:
        print(f"torch.compile skipped ({e})")

In [ ]:
from scipy.optimize import linear_sum_assignment

def to_vec(ang):
    th, ph = ang[...,0], ang[...,1]
    return torch.stack([torch.sin(th)*torch.cos(ph),
                         torch.sin(th)*torch.sin(ph),
                         torch.cos(th)], dim=-1)

def ang_sq(p, t):
    return ((to_vec(p) - to_vec(t))**2).sum(-1)

@torch.no_grad()
def hungarian_match(out, tgt_tracks, tgt_masks):
    B  = out['pos'].shape[0]
    pp = out['pos'].detach().cpu()
    pa = out['ang'].detach().cpu()
    pe = out['exist'].detach().cpu().squeeze(-1)
    assignments = []
    for b in range(B):
        valid = tgt_masks[b].cpu(); T = int(valid.sum())
        if T == 0:
            assignments.append((np.array([],int), np.array([],int))); continue
        tp = tgt_tracks[b,valid,:2].cpu()
        ta = tgt_tracks[b,valid,2:].cpu()
        K  = MAX_TRACKS
        pp_b = pp[b].unsqueeze(1).expand(-1,T,-1)
        pa_b = pa[b].unsqueeze(1).expand(-1,T,-1)
        C = (W_POS  * (pp_b - tp.unsqueeze(0).expand(K,-1,-1)).abs().sum(-1)
           + W_ANG  * ang_sq(pa_b, ta.unsqueeze(0).expand(K,-1,-1))
           - torch.sigmoid(pe[b]).unsqueeze(1).expand(-1,T))
        ri, ci = linear_sum_assignment(C.numpy())
        valid_idx = torch.where(valid)[0].numpy()
        assignments.append((ri, valid_idx[ci]))
    return assignments


class AcousticDETRLoss(nn.Module):
    def forward(self, out, tgt_tracks, tgt_masks):
        B, K = out['pos'].shape[:2]; dev = out['pos'].device
        tgt_tracks = tgt_tracks.to(dev); tgt_masks = tgt_masks.to(dev)
        assignments = hungarian_match(out, tgt_tracks, tgt_masks)
        l_pos = l_ang = torch.tensor(0., device=dev); n = 0
        for b, (pi, ti) in enumerate(assignments):
            if not len(pi): continue
            pi_t = torch.tensor(pi, device=dev); ti_t = torch.tensor(ti, device=dev)
            l_pos = l_pos + F.mse_loss(out['pos'][b][pi_t], tgt_tracks[b,ti_t,:2])
            l_ang = l_ang + ang_sq(out['ang'][b][pi_t], tgt_tracks[b,ti_t,2:]).mean()
            n += len(pi)
        if n > 0: l_pos /= B; l_ang /= B
        exist_tgt = torch.zeros(B, K, device=dev)
        w_e       = torch.full((B,K), NO_TRACK_COEFF, device=dev)
        for b, (pi, _) in enumerate(assignments):
            if len(pi): exist_tgt[b,pi]=1.; w_e[b,pi]=1.
        l_exist = F.binary_cross_entropy_with_logits(
            out['exist'].squeeze(-1), exist_tgt, weight=w_e)
        total = W_POS*l_pos + W_ANG*l_ang + W_EXIST*l_exist
        return total, {'total':total.item(),'pos':l_pos.item(),
                       'ang':l_ang.item(),'exist':l_exist.item()}, assignments

criterion = AcousticDETRLoss()
print("Loss ready ")

In [ ]:
from torch.cuda.amp import GradScaler, autocast
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython.display import clear_output

torch.backends.cudnn.benchmark = True

# ── Optimiser ─────────────────────────────────────────────────────────────────
optimiser = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=1e-4,
    fused=(DEVICE.type=='cuda'))

def lr_lambda(ep):
    if ep < WARMUP_EPOCHS: return (ep+1)/WARMUP_EPOCHS
    p = (ep - WARMUP_EPOCHS) / max(N_EPOCHS - WARMUP_EPOCHS, 1)
    return 0.5*(1 + math.cos(math.pi*p))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda)
scaler    = GradScaler(enabled=USE_AMP and DEVICE.type=='cuda')

# ── Resume from checkpoint ────────────────────────────────────────────────────
CKPT  = os.path.join(SAVE_DIR, 'best_model.pt')
START = 1; best_val = float('inf')
history = {k:[] for k in ['tr','va','ang','pos','eff','fake']}

if os.path.exists(CKPT):
    print("Resuming from checkpoint …")
    ckpt = torch.load(CKPT, map_location=DEVICE)
    base = getattr(model, '_orig_mod', model)
    base.load_state_dict(ckpt['model'])
    optimiser.load_state_dict(ckpt['opt'])
    scheduler.load_state_dict(ckpt['sched'])
    START    = ckpt['epoch'] + 1
    best_val = ckpt['best_val']
    history  = ckpt.get('history', history)
    print(f"  Epoch {START}, best_val={best_val:.5f}")

# ── Metric helper ─────────────────────────────────────────────────────────────
def batch_metrics(out, tgt_tracks, tgt_masks, assignments):
    pprob = out['prob'].squeeze(-1).detach().cpu()
    pp    = out['pos'].detach().cpu().numpy()
    pa    = out['ang'].detach().cpu().numpy()
    tt    = tgt_tracks.cpu().numpy()
    tm    = tgt_masks.cpu().numpy()
    angs, poss, n_true_total, n_matched, n_pred = [], [], int(tm.sum()), 0, 0

    for b, (pi, ti) in enumerate(assignments):
        n_pred += int((pprob[b] > EXIST_THRESH).sum().item())
        for p_i, t_i in zip(pi, ti):
            def v(a):
                return np.array([math.sin(a[0])*math.cos(a[1]),
                                  math.sin(a[0])*math.sin(a[1]), math.cos(a[0])])
            cos_a = np.clip(np.dot(v(pa[b,p_i]), v(tt[b,t_i,2:])), -1, 1)
            angs.append(math.degrees(math.acos(cos_a)))
            poss.append(np.linalg.norm(pp[b,p_i] - tt[b,t_i,:2]) * 1e3)
            n_matched += 1

    eff  = n_matched / max(n_true_total, 1)
    fake = max(0, n_pred - n_matched) / max(n_pred, 1)
    return (np.nanmean(angs) if angs else float('nan'),
            np.nanmean(poss) if poss else float('nan'), eff, fake)

# ── Live plot ─────────────────────────────────────────────────────────────────
def live_plot(h, ep):
    clear_output(wait=True)
    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    fig.suptitle(f'AcousticDETR — Epoch {ep}/{N_EPOCHS}', fontsize=13)
    eps = range(1, len(h['tr'])+1)
    for ax, (k, lbl, col) in zip(axes.flat, [
        ('tr','Train loss','tab:blue'), ('va','Val loss','tab:orange'),
        ('ang','Angular err (°)','tab:green'), ('pos','Position err (mm)','tab:red'),
        ('eff','Efficiency','tab:purple'), ('fake','Fake rate','tab:brown')]):
        if h[k]: ax.plot(eps, h[k], color=col, lw=1.5)
        ax.set_title(lbl, fontsize=10); ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR,'training_curves.png'), dpi=100)
    plt.show()

# ── Main loop ─────────────────────────────────────────────────────────────────
print(f"Training: {N_EPOCHS} epochs | batch={BATCH_SIZE} | AMP={USE_AMP}")
print(f"{'Ep':>4}  {'TrLoss':>7}  {'VaLoss':>7}  {'Ang°':>6}  {'Pos mm':>7}  {'Eff%':>5}  {'Fake%':>5}")
print('─'*55)

for ep in range(START, N_EPOCHS+1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train(); tr_loss = 0.
    optimiser.zero_grad(set_to_none=True)

    for step, (wf, trk, msk, _) in enumerate(train_dl):
        wf  = wf.to(DEVICE,  non_blocking=True)
        trk = trk.to(DEVICE, non_blocking=True)
        msk = msk.to(DEVICE, non_blocking=True)

        with autocast(enabled=USE_AMP and DEVICE.type=='cuda'):
            loss, _, _ = criterion(model(wf), trk, msk)
            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step+1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimiser)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimiser); scaler.update()
            optimiser.zero_grad(set_to_none=True)

        tr_loss += loss.item() * GRAD_ACCUM_STEPS * len(wf)

    tr_loss /= n_train; scheduler.step()

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval(); va_loss = 0.
    A, P, E, FK = [], [], [], []

    with torch.no_grad():
        for wf, trk, msk, _ in val_dl:
            wf  = wf.to(DEVICE,  non_blocking=True)
            trk = trk.to(DEVICE, non_blocking=True)
            msk = msk.to(DEVICE, non_blocking=True)
            with autocast(enabled=USE_AMP and DEVICE.type=='cuda'):
                out = model(wf); loss, _, assn = criterion(out, trk, msk)
            va_loss += loss.item() * len(wf)
            a,p,e,f = batch_metrics(out, trk, msk, assn)
            A.append(a); P.append(p); E.append(e); FK.append(f)

    va_loss /= n_val
    am, pm, em, fm = (float(np.nanmean(x)) for x in [A,P,E,FK])

    for k,v in zip(['tr','va','ang','pos','eff','fake'],
                    [tr_loss,va_loss,am,pm,em,fm]):
        history[k].append(v)

    tag = ''
    if va_loss < best_val:
        best_val = va_loss
        base = getattr(model,'_orig_mod',model)
        torch.save({'epoch':ep,'model':base.state_dict(),
                    'opt':optimiser.state_dict(),'sched':scheduler.state_dict(),
                    'best_val':best_val,'history':history}, CKPT)
        tag = ' ✓'

    if ep%5==0 or ep==1:
        print(f"{ep:>4}  {tr_loss:>7.4f}  {va_loss:>7.4f}  "
              f"{am:>6.2f}  {pm:>7.2f}  {em*100:>5.1f}  {fm*100:>5.1f}{tag}")

    if ep%10==0: live_plot(history, ep)

print(f"\nDone. Best val loss: {best_val:.5f}  → {CKPT}")
live_plot(history, N_EPOCHS)

In [ ]:
from collections import defaultdict

ckpt = torch.load(CKPT, map_location=DEVICE)
base = getattr(model,'_orig_mod',model)
base.load_state_dict(ckpt['model']); model.eval()
print(f"Best checkpoint: epoch {ckpt['epoch']}, val_loss={ckpt['best_val']:.5f}")

results = defaultdict(list)
conf_mat = np.zeros((5,5), int)

with torch.no_grad():
    for wf, trk, msk, ntrk in val_dl:
        wf  = wf.to(DEVICE, non_blocking=True)
        trk = trk.to(DEVICE, non_blocking=True)
        msk = msk.to(DEVICE, non_blocking=True)
        out  = model(wf)
        assn = hungarian_match(out, trk, msk)
        pp   = out['pos'].cpu().numpy()
        pa   = out['ang'].cpu().numpy()
        prob = out['prob'].squeeze(-1).cpu()
        tt   = trk.cpu().numpy(); tm = msk.cpu().numpy()

        for b,(pi,ti) in enumerate(assn):
            nt = int(tm[b].sum())
            np_ = int((prob[b]>EXIST_THRESH).sum())
            conf_mat[min(nt,4), min(np_,4)] += 1
            for p_i,t_i in zip(pi,ti):
                def v(a):
                    return np.array([math.sin(a[0])*math.cos(a[1]),
                                      math.sin(a[0])*math.sin(a[1]), math.cos(a[0])])
                cos_a = np.clip(np.dot(v(pa[b,p_i]), v(tt[b,t_i,2:])), -1, 1)
                results[nt].append({
                    'ang': math.degrees(math.acos(cos_a)),
                    'pos': np.linalg.norm(pp[b,p_i]-tt[b,t_i,:2])*1e3,
                    'theta_true': tt[b,t_i,2], 'true_n':nt, 'pred_n':np_})

all_r = [d for lst in results.values() for d in lst]

print("\n" + "="*62)
print(f"{'Metric':<28} {'Overall':>8}  k=1   k=2   k=3   k=4")
print('─'*62)
def mean_m(lst, key): return np.nanmean([d[key] for d in lst]) if lst else float('nan')
for key, label in [('ang','Angular res. (°)'), ('pos','Pos. res. (mm)')]:
    row = f"{label:<28} {mean_m(all_r,key):>8.3f}"
    for k in range(1,5): row += f"  {mean_m(results[k],key):>5.3f}"
    print(row)
eff_row = f"{'Efficiency (%)':<28} {np.nanmean([d['pred_n']>0 for d in all_r])*100:>8.1f}"
for k in range(1,5): eff_row += f"  {len(results[k])/max(sum(1 for d in all_r if d['true_n']==k),1)*100:>5.1f}"
print(eff_row)
print("="*62)

print("\nMultiplicity confusion (rows=true, cols=pred):")
print("     " + "  ".join(f"P={j}" for j in range(5)))
for i in range(5): print(f"T={i}  " + "  ".join(f"{conf_mat[i,j]:4d}" for j in range(5)))

## Event display

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

COLORS = ['#E63946','#457B9D','#2A9D8F','#E9C46A']
CMAPS  = ['Reds','Blues','Greens','Oranges']

def hits_all_sheets(x0,y0,theta,phi):
    return np.array([hit_xy(x0,y0,theta,phi,k) for k in range(N_SHEETS)])

def gaussian_blob(cx,cy,res=100,sigma=0.012):
    half=SHEET_SIZE/2; xs=np.linspace(-half,half,res); ys=np.linspace(-half,half,res)
    XX,YY=np.meshgrid(xs,ys)
    return np.exp(-((XX-cx)**2+(YY-cy)**2)/(2*sigma**2)), xs, ys

def plot_event(pred_tracks, true_tracks=None, waveforms=None, title='AcousticDETR', save=None):
    n_rows = 3 if waveforms is not None else 2
    fig    = plt.figure(figsize=(4*N_SHEETS, 3.8*n_rows))
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)
    half   = SHEET_SIZE/2; z0,z1 = -SHEET_GAP, N_SHEETS*SHEET_GAP

    # ── 3-D view ────────────────────────────────────────────────────────────
    ax3 = fig.add_subplot(n_rows,1,1, projection='3d')
    for k in range(N_SHEETS):
        z=k*SHEET_GAP
        ax3.add_collection3d(Poly3DCollection(
            [[[-half,-half,z],[half,-half,z],[half,half,z],[-half,half,z]]],
            alpha=0.07, facecolor='lightgrey', edgecolor='grey', lw=0.5))
        ax3.scatter(MIC_XY[:,0],MIC_XY[:,1],np.full(M_MICS,z),s=12,c='grey',marker='^')
        ax3.text(half*1.05,0,z,f'S{k}',fontsize=7,color='grey')

    def draw3d(x0,y0,th,ph,color,ls,lw,label):
        dz=math.cos(th); dx=math.sin(th)*math.cos(ph); dy=math.sin(th)*math.sin(ph)
        if abs(dz)<1e-9: return
        ax3.plot([x0+z0/dz*dx,x0+z1/dz*dx],[y0+z0/dz*dy,y0+z1/dz*dy],[z0,z1],
                 color=color,ls=ls,lw=lw,label=label)
        ax3.scatter([x0],[y0],[0],color=color,s=50,zorder=10)

    if true_tracks:
        for i,t in enumerate(true_tracks):
            draw3d(t['x0'],t['y0'],t['theta'],t['phi'],'black','--',1.5,f'True {i+1}')
    for i,t in enumerate(pred_tracks):
        draw3d(t['x0'],t['y0'],t['theta'],t['phi'],COLORS[i%4],'-',2.5,
               f"Pred {i+1} (p={t['conf']:.2f})")
    ax3.set_xlabel('X[m]'); ax3.set_ylabel('Y[m]'); ax3.set_zlabel('Z[m]')
    ax3.set_xlim(-half*1.3,half*1.3); ax3.set_ylim(-half*1.3,half*1.3); ax3.set_zlim(z0,z1)
    ax3.legend(fontsize=8,loc='upper left'); ax3.set_title('3-D Track Reconstruction')

    # ── Sheet heatmaps ──────────────────────────────────────────────────────
    axh = [fig.add_subplot(n_rows,N_SHEETS,N_SHEETS+k+1) for k in range(N_SHEETS)]
    for k,ax in enumerate(axh):
        ax.set_aspect('equal'); ax.set_facecolor('#f4f4f4')
        for ti,tr in enumerate(pred_tracks):
            cx,cy = hits_all_sheets(tr['x0'],tr['y0'],tr['theta'],tr['phi'])[k]
            blob,xs,ys = gaussian_blob(cx,cy)
            ax.contourf(xs,ys,blob*tr['conf'],levels=5,
                        cmap=plt.get_cmap(CMAPS[ti%4]),alpha=0.55,vmin=0,vmax=1)
        if true_tracks:
            for tr in true_tracks:
                cx,cy = hits_all_sheets(tr['x0'],tr['y0'],tr['theta'],tr['phi'])[k]
                ax.plot(cx,cy,'k+',ms=10,mew=2)
        ax.scatter(MIC_XY[:,0],MIC_XY[:,1],s=18,c='dimgrey',marker='^',zorder=5)
        ax.add_patch(plt.Rectangle((-half,-half),SHEET_SIZE,SHEET_SIZE,
                                    fill=False,edgecolor='black',lw=1.5))
        ax.set_xlim(-half*1.1,half*1.1); ax.set_ylim(-half*1.1,half*1.1)
        ax.set_title(f'Sheet {k}',fontsize=9); ax.tick_params(labelsize=6)
    patches = [mpatches.Patch(color=plt.get_cmap(CMAPS[i%4])(0.7),
               label=f"Track {i+1} (p={t['conf']:.2f})")
               for i,t in enumerate(pred_tracks)]
    if true_tracks: patches.append(mpatches.Patch(color='black',label='True hit (+)'))
    axh[0].legend(handles=patches,fontsize=7); axh[0].set_ylabel('Y[m]',fontsize=7)

    # ── Raw waveforms ────────────────────────────────────────────────────────
    if waveforms is not None:
        t_us = np.arange(WAVEFORM_T)/SAMPLE_RATE*1e6
        for m in range(M_MICS):
            ax=fig.add_subplot(n_rows,M_MICS,2*N_SHEETS+m+1)
            for k in range(N_SHEETS): ax.plot(t_us,waveforms[k,m],alpha=0.6,lw=0.7,label=f'S{k}')
            ax.set_title(f'Mic {m}',fontsize=8); ax.set_xlabel('µs',fontsize=7)
            if m==0: ax.set_ylabel('Amp',fontsize=7); ax.legend(fontsize=6)
            ax.tick_params(labelsize=6)

    plt.tight_layout()
    if save: plt.savefig(save,dpi=140,bbox_inches='tight'); print(f"Saved → {save}")
    plt.show()


model.eval()
for n_pileup in [1, 2, 3, 4]:
    ev   = simulate_one(n_pileup*77 + 3)
    wf_t = torch.tensor(ev[0][None]).to(DEVICE)
    with torch.no_grad():
        pred = model.predict(wf_t)[0]
    true = [{'x0':ev[1][i,0],'y0':ev[1][i,1],'theta':ev[1][i,2],'phi':ev[1][i,3]}
             for i in range(n_pileup)]
    print(f"n_true={n_pileup}  n_pred={len(pred)}")
    plot_event(pred, true, ev[0],
               title=f'AcousticDETR — {n_pileup}-track pileup event',
               save=os.path.join(SAVE_DIR,f'event_k{n_pileup}.png'))